In the previous notebook, We built a single agent that could take action. Now, we'll learn how to scale up by building agent teams.

Just like a team of people, we can create specialized agents that collaborate to solve complex problems. This is called a multi-agent system, and it's one of the most powerful concepts in AI agent development.

In [11]:
%pip install google-adk


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
from dotenv import load_dotenv, find_dotenv, dotenv_values
from pathlib import Path
import os

cwd = Path.cwd()

candidates = [
    cwd / ".env",
    cwd.parent / ".env",
]

loaded = None
for c in candidates:
    if c.exists():
        load_dotenv(c, override=True)
        loaded = c
        break

if loaded is None:
    # fallback: search upwards from current working directory
    found = find_dotenv(usecwd=True)
    if found:
        load_dotenv(found, override=True)
        loaded = Path(found)

if loaded is None:
    raise FileNotFoundError(f"No .env found. Tried: {[str(c) for c in candidates]} and find_dotenv")

# Byte level debug (reveals BOM / CRLF / hidden chars)
raw = loaded.read_bytes()[:80]
print("first_bytes_repr=", repr(raw))

parsed = dotenv_values(str(loaded))

print("cwd=", str(cwd))
print("loaded_env=", str(loaded))
print("parsed_keys=", sorted([k for k in parsed.keys() if k]))
print("dotenv_has_GOOGLE_API_KEY=", "GOOGLE_API_KEY" in parsed)

# Ensure current process env is updated
load_dotenv(str(loaded), override=True)

print("process_has_GOOGLE_API_KEY=", "GOOGLE_API_KEY" in os.environ)
print("value_is_none=", os.getenv("GOOGLE_API_KEY") is None)
print("value_length=", None if os.getenv("GOOGLE_API_KEY") is None else len(os.getenv("GOOGLE_API_KEY")))

first_bytes_repr= b'GOOGLE_GENAI_USE_VERTEXAI=0\nGOOGLE_API_KEY=AIzaSyDUtWExUnLHnqYAJ_4iUyVn8hdGQbf-0'
cwd= /Users/marziehfattahi/Documents/New project/resume-engineering-portfolio/Lernen/AI Agents Intensive Course with Google/Day 1.b
loaded_env= /Users/marziehfattahi/Documents/New project/resume-engineering-portfolio/Lernen/AI Agents Intensive Course with Google/Day 1.b/.env
parsed_keys= ['GOOGLE_API_KEY', 'GOOGLE_GENAI_USE_VERTEXAI']
dotenv_has_GOOGLE_API_KEY= True
process_has_GOOGLE_API_KEY= True
value_is_none= False
value_length= 39


In [20]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [21]:
retry_config=types.HttpRetryOptions(
    # Maximum retry attempts
    attempts=5,  
    # Delay multiplier
    exp_base=7,  
    initial_delay=1,
    # Retry on these HTTP errors
    http_status_codes=[429, 500, 503, 504], 
)

### Why Multi-Agent Systems? ¶
The Problem: The "Do-It-All" Agent

Single agents can do a lot. But what happens when the task gets complex? A single "monolithic" agent that tries to do research, writing, editing, and fact-checking all at once becomes a problem. Its instruction prompt gets long and confusing. It's hard to debug (which part failed?), difficult to maintain, and often produces unreliable results.

The Solution: A Team of Specialists

Instead of one "do-it-all" agent, we can build a multi-agent system. This is a team of simple, specialized agents that collaborate, just like a real-world team. Each agent has one clear job (e.g., one agent only does research, another only writes). This makes them easier to build, easier to test, and much more powerful and reliable when working together.



In [22]:
research_agent = Agent(
    name="ResearchAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a specialized research agent. Your only job is to use the
    google_search tool to find 2-3 pieces of relevant information on the given topic and present the findings with citations.""",
    tools=[google_search],
    output_key="research_findings",  # The result of this agent will be stored in the session state with this key.
)

print("✅ research_agent created.")

✅ research_agent created.


In [23]:
# Summarizer Agent: Its job is to summarize the text it receives.
summarizer_agent = Agent(
    name="SummarizerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request a bulleted list for a clear output format.
    instruction="""Read the provided research findings: {research_findings}
Create a concise summary as a bulleted list with 3-5 key points.""",
    output_key="final_summary",
)

print("✅ summarizer_agent created.")

✅ summarizer_agent created.


In [24]:
root_agent = Agent(
    name="ResearchCoordinator",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This instruction tells the root agent HOW to use its tools (which are the other agents).
    instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the `ResearchAgent` tool to find relevant information on the topic provided by the user.
2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
3. Finally, present the final summary clearly to the user as your response.""",
    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.
    tools=[AgentTool(research_agent), AgentTool(summarizer_agent)],
)

print("✅ root_agent created.")

✅ root_agent created.


In [25]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "What are the latest advancements in quantum computing and what do they mean for AI?"
)


 ### Created new session: debug_session_id

User > What are the latest advancements in quantum computing and what do they mean for AI?


ResearchCoordinator > Recent advancements in quantum computing are poised to significantly impact artificial intelligence by offering solutions to complex computational challenges. Key developments include:

*   **Enhanced Computational Power and Efficiency:** Quantum computers can perform parallel processing at speeds unattainable by classical computers, which is expected to drastically speed up AI model training, especially in deep learning. Furthermore, quantum AI models hold the promise of being significantly more energy-efficient than their classical counterparts.
*   **Improved Problem-Solving Capabilities:** Quantum algorithms are showing promise in improving AI's ability to handle complex optimization tasks and process large datasets more efficiently. This could unlock breakthroughs in fields like drug discovery and materials science, which are currently limited by classical computing power.
*   **Development of Advanced AI Models:** The emergence of quantum machine learning (Q

### Sequential Workflows - The Assembly Line¶
The Problem: Unpredictable Order

The previous multi-agent system worked, but it relied on a detailed instruction prompt to force the LLM to run steps in order. This can be unreliable. A complex LLM might decide to skip a step, run them in the wrong order, or get "stuck," making the process unpredictable.

The Solution: A Fixed Pipeline

When you need tasks to happen in a guaranteed, specific order, you can use a SequentialAgent. This agent acts like an assembly line, running each sub-agent in the exact order you list them. The output of one agent automatically becomes the input for the next, creating a predictable and reliable workflow.

Use Sequential when: Order matters, you need a linear pipeline, or each step builds on the previous one.

In [26]:
outline_agent = Agent(
    name="OutlineAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Create a blog outline for the given topic with:
    1. A catchy headline
    2. An introduction hook
    3. 3-5 main sections with 2-3 bullet points for each
    4. A concluding thought""",
    output_key="blog_outline",  
)
# The result of this agent will be stored in the session state with this key.
print("✅ outline_agent created.")

✅ outline_agent created.


In [27]:
writer_agent = Agent(
    name="WriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Following this outline strictly: {blog_outline}
    Write a brief, 200 to 300-word blog post with an engaging and informative tone.""",
    output_key="blog_draft",  
)

print("✅ writer_agent created.")

✅ writer_agent created.


In [28]:
editor_agent = Agent(
    name="EditorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Edit this draft: {blog_draft}
    Your task is to polish the text by fixing any grammatical errors, improving the flow and sentence structure, and enhancing overall clarity.""",
    output_key="final_blog",  
)

print("✅ editor_agent created.")

✅ editor_agent created.


In [29]:
root_agent = SequentialAgent(
    name="BlogPipeline",
    sub_agents=[outline_agent, writer_agent, editor_agent],
)

print("✅ Sequential Agent created.")

✅ Sequential Agent created.


In [30]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a blog post about the benefits of multi-agent systems for software developers"
)


 ### Created new session: debug_session_id

User > Write a blog post about the benefits of multi-agent systems for software developers
OutlineAgent > ## Blog Outline: Multi-Agent Systems: Your Software's New Superpower

**Introduction Hook:**

*   Imagine a team of highly specialized, independent software "agents" working in unison to achieve complex goals. This isn't science fiction; it's the power of multi-agent systems (MAS), and it's poised to revolutionize how we build software.

**Main Sections:**

**1. Enhanced Problem Solving & Complexity Management**

*   **Decomposition is Key:** MAS allows complex problems to be broken down into smaller, more manageable tasks handled by individual, specialized agents. This simplifies development and debugging.
*   **Emergent Behavior:** Observe how simple agent interactions can lead to sophisticated, intelligent system-level behavior that would be incredibly difficult to program directly.
*   **Adaptability and Resilience:** If one agent fa

### Parallel Workflows - Independent Researchers¶
The Problem: The Bottleneck

The previous sequential agent is great, but it's an assembly line. Each step must wait for the previous one to finish. What if you have several tasks that are not dependent on each other? For example, researching three different topics. Running them in sequence would be slow and inefficient, creating a bottleneck where each task waits unnecessarily.

The Solution: Concurrent Execution

When you have independent tasks, you can run them all at the same time using a ParallelAgent. This agent executes all of its sub-agents concurrently, dramatically speeding up the workflow. Once all parallel tasks are complete, you can then pass their combined results to a final 'aggregator' step.

Use Parallel when: Tasks are independent, speed matters, and you can execute concurrently.

In [31]:
tech_researcher = Agent(
    name="TechResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research the latest AI/ML trends. Include 3 key developments,
the main companies involved, and the potential impact. Keep the report very concise (100 words).""",
    tools=[google_search],
    output_key="tech_research",  
)

print("✅ tech_researcher created.")

✅ tech_researcher created.


In [32]:
health_researcher = Agent(
    name="HealthResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research recent medical breakthroughs. Include 3 significant advances,
their practical applications, and estimated timelines. Keep the report concise (100 words).""",
    tools=[google_search],
    output_key="health_research",  
)

In [33]:
finance_researcher= Agent(
    name= "FinanceResearcher",
    model = Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research current fintech trends. Include 3 key trends,
their market implications, and the future outlook. Keep the report concise (100 words).""",
    tools=[google_search],
    output_key="finance_research",
)

print("✅ finance_researcher created.")



✅ finance_researcher created.


In [34]:
aggregator_agent = Agent(
    name="AggregatorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    
    instruction="""Combine these three research findings into a single executive summary:

    **Technology Trends:**
    {tech_research}
    
    **Health Breakthroughs:**
    {health_research}
    
    **Finance Innovations:**
    {finance_research}
    
    Your summary should highlight common themes, surprising connections, and the most important key takeaways from all three reports. The final summary should be around 200 words.""",
    output_key="executive_summary", 
)

print("✅ aggregator_agent created.")

✅ aggregator_agent created.


Then we bring the agents together under a parallel agent, which is itself nested inside of a sequential agent.

In [35]:
parallel_research_team = ParallelAgent(
    name="ParallelResearchTeam",
    sub_agents=[tech_researcher, health_researcher, finance_researcher],
)

# This SequentialAgent defines the high-level workflow: run the parallel team first, then run the aggregator.
root_agent = SequentialAgent(
    name="ResearchSystem",
    sub_agents=[parallel_research_team, aggregator_agent],
)

print("✅ Parallel and Sequential Agents created.")

✅ Parallel and Sequential Agents created.


In [36]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Run the daily executive briefing on Tech, Health, and Finance"
)


 ### Created new session: debug_session_id

User > Run the daily executive briefing on Tech, Health, and Finance
TechResearcher > Here are three key AI/ML trends for 2026:

1.  **Autonomous AI Agents:** These agents will operate with significant autonomy, capable of planning, reasoning, and executing complex tasks with minimal human intervention. Companies like Google (DeepMind) and Microsoft are at the forefront of developing these agentic systems. The impact will be increased automation of multi-step processes, driving efficiency and innovation across various business functions.

2.  **Multimodal AI:** AI systems will increasingly process and integrate multiple types of data simultaneously (text, voice, image, etc.) to understand context better and provide more nuanced responses. This is becoming the default standard for intelligent systems. Major players like Google and Microsoft are heavily involved. The impact will be more natural and efficient human-AI interactions, leading to b

### Loop Workflows - The Refinement Cycle¶
The Problem: One-Shot Quality

All the workflows we've seen so far run from start to finish. The SequentialAgent and ParallelAgent produce their final output and then stop. This 'one-shot' approach isn't good for tasks that require refinement and quality control. What if the first draft of our story is bad? We have no way to review it and ask for a rewrite.

The Solution: Iterative Refinement

When a task needs to be improved through cycles of feedback and revision, you can use a LoopAgent. A LoopAgent runs a set of sub-agents repeatedly until a specific condition is met or a maximum number of iterations is reached. This creates a refinement cycle, allowing the agent system to improve its own work over and over.

Use Loop when: Iterative improvement is needed, quality refinement matters, or you need repeated cycles.

In [37]:
initial_writer_agent = Agent(
    name="InitialWriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Based on the user's prompt, write the first draft of a short story (around 100-150 words).
    Output only the story text, with no introduction or explanation.""",
    output_key="current_story",  # Stores the first draft in the state.
)

print("✅ initial_writer_agent created.")

✅ initial_writer_agent created.


In [38]:
critic_agent = Agent(
    name="CriticAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a constructive story critic. Review the story provided below.
    Story: {current_story}
    
    Evaluate the story's plot, characters, and pacing.
    - If the story is well-written and complete, you MUST respond with the exact phrase: "APPROVED"
    - Otherwise, provide 2-3 specific, actionable suggestions for improvement.""",
    output_key="critique",  # Stores the feedback in the state.
)

print("✅ critic_agent created.")

✅ critic_agent created.


In [39]:
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the story is finished and no more changes are needed."""
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


print("✅ exit_loop function created.")

✅ exit_loop function created.


In [40]:
refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a story refiner. You have a story draft and critique.
    
    Story Draft: {current_story}
    Critique: {critique}
    
    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", you MUST call the `exit_loop` function and nothing else.
    - OTHERWISE, rewrite the story draft to fully incorporate the feedback from the critique.""",
    output_key="current_story",  
    tools=[
        FunctionTool(exit_loop)
    ], 
)

print("✅ refiner_agent created.")

✅ refiner_agent created.


In [41]:
story_refinement_loop = LoopAgent(
    name="StoryRefinementLoop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=2,  # Prevents infinite loops
)


root_agent = SequentialAgent(
    name="StoryPipeline",
    sub_agents=[initial_writer_agent, story_refinement_loop],
)

print("✅ Loop and Sequential Agents created.")

✅ Loop and Sequential Agents created.


In [42]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a short story about a lighthouse keeper who discovers a mysterious, glowing map"
)


 ### Created new session: debug_session_id

User > Write a short story about a lighthouse keeper who discovers a mysterious, glowing map
InitialWriterAgent > Elias, his beard flecked with salt and time, polished the Fresnel lens with practiced, weary hands. The foghorn’s mournful groan was his lullaby. Tonight, however, a different sound pricked his ears – a soft, rhythmic tapping against the lantern room glass. He peered out, expecting driftwood, but found a rolled parchment, tied with a luminescent, seaweed-like cord.

Unfurling it, Elias gasped. The parchment pulsed with an inner light, its markings not ink, but shimmering constellations he’d never seen. Strange symbols, reminiscent of ancient script, wove across the glowing surface, hinting at unknown shores and forgotten paths. It wasn’t a map of the sea he knew, but of something far more… celestial.
CriticAgent > This is a promising start to a mystery! Here are a few suggestions for improvement:

*   **Character Depth:** Elias i